In [1]:
from pathlib import Path
import pandas as pd
import yaml

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
print("Repo root:", ROOT)
cfg = yaml.safe_load(open(ROOT / "config.yaml", encoding="utf-8"))
RAW = ROOT / "data" / "raw"

# Only the data files (inside conjunto_de_datos), only the two parts we need
denue_files = sorted(
    f for f in RAW.rglob("conjunto_de_datos/*.csv")
    if f.name.startswith(("denue_inegi_46111", "denue_inegi_46112"))
)
for f in denue_files:
    print(f.name, round(f.stat().st_size / 1e6), "MB")

peek = pd.read_csv(denue_files[0], encoding="latin-1", dtype=str, nrows=5)
print(list(peek.columns))
peek

Repo root: C:\Users\nbeckers\Documents\Naomi\PG Diploma in DS and AI\Pillar V\abarrotes-micromarket-segmentation
denue_inegi_46111_.csv 369 MB
denue_inegi_46112-46311_.csv 318 MB
['id', 'clee', 'nom_estab', 'raz_social', 'codigo_act', 'nombre_act', 'per_ocu', 'tipo_vial', 'nom_vial', 'tipo_v_e_1', 'nom_v_e_1', 'tipo_v_e_2', 'nom_v_e_2', 'tipo_v_e_3', 'nom_v_e_3', 'numero_ext', 'letra_ext', 'edificio', 'edificio_e', 'numero_int', 'letra_int', 'tipo_asent', 'nomb_asent', 'tipoCenCom', 'nom_CenCom', 'num_local', 'cod_postal', 'cve_ent', 'entidad', 'cve_mun', 'municipio', 'cve_loc', 'localidad', 'ageb', 'manzana', 'telefono', 'correoelec', 'www', 'tipoUniEco', 'latitud', 'longitud', 'fecha_alta']


,id,clee,nom_estab,raz_social,codigo_act,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,...,localidad,ageb,manzana,telefono,correoelec,www,tipoUniEco,latitud,longitud,fecha_alta
0,12312018,01001461110072141000000000U1,NaN,NaN,461110,"Comercio al por menor en tiendas de abarrotes,...",0 a 5 personas,AVENIDA,NINGUNO,PRIVADA,...,Aguascalientes,0892,039,NaN,NaN,NaN,Fijo,21.87371282,-102.2851598,2026-04
1,10060248,01001461110065211000000000U1,2BOX,2BOX,461110,"Comercio al por menor en tiendas de abarrotes,...",0 a 5 personas,BOULEVARD,BOULEVARD A ZACATECAS,CALLE,...,Aguascalientes,1975,002,NaN,NaN,NaN,Fijo,21.91162498,-102.29200366,2024-11
2,9604852,01001461110069191000000000U1,2BOX,ALPEZ NBG,461110,"Comercio al por menor en tiendas de abarrotes,...",0 a 5 personas,AVENIDA,QUINTA,CALLE,...,Aguascalientes,0869,019,3339523807,CAPAGAS@2BOX.MX,NaN,Fijo,21.86546617,-102.2982667,2024-11
3,10550501,01001461110065751000000000U6,3 CACHORROS,MAURICIO ROCHA,461110,"Comercio al por menor en tiendas de abarrotes,...",0 a 5 personas,CALLE,OSCAR HERNANDEZ DUQUE MEDINA,CALLE,...,Aguascalientes,1161,026,NaN,NaN,NaN,Fijo,21.80667148,-102.27252245,2024-11
4,6988165,01001461130012911010000000U8,ABAROTES DANI,NaN,461110,"Comercio al por menor en tiendas de abarrotes,...",0 a 5 personas,CALLE,MARCOS CORONADO,CALLE,...,Pocitos ...,3948,034,NaN,NaN,NaN,Fijo,21.92729578,-102.33545683,2019-11


In [2]:
states = cfg["region"]["states"]          # ["09", "15", "13"]
keep_cols = ["id", "clee", "nom_estab", "codigo_act", "nombre_act", "per_ocu",
             "tipo_asent", "nomb_asent", "tipoCenCom", "cod_postal",
             "cve_ent", "entidad", "cve_mun", "municipio", "cve_loc", "localidad",
             "ageb", "manzana", "tipoUniEco", "latitud", "longitud", "fecha_alta"]

parts = []
for f in denue_files:
    for chunk in pd.read_csv(f, encoding="latin-1", dtype=str,
                             usecols=keep_cols, chunksize=200_000):
        parts.append(chunk[chunk["cve_ent"].str.zfill(2).isin(states)])
    print("done:", f.name)

denue = pd.concat(parts, ignore_index=True)
out = ROOT / "data" / "interim" / "denue_zmvm_retail.parquet"
denue.to_parquet(out, index=False)

print("\nRows kept:", len(denue), "| saved to", out.name)
print("\nAll food retail by state:\n", denue.groupby("entidad").size())
abarrotes = denue[denue["codigo_act"] == "461110"]
print("\nTiendas de abarrotes by state:\n", abarrotes.groupby("entidad").size())
print("\nTop activities:\n", denue["nombre_act"].value_counts().head(10))

done: denue_inegi_46111_.csv
done: denue_inegi_46112-46311_.csv

Rows kept: 331093 | saved to denue_zmvm_retail.parquet

All food retail by state:
 entidad
Ciudad de México     96783
Hidalgo              32630
México              201680
dtype: int64

Tiendas de abarrotes by state:
 entidad
Ciudad de México     38996
Hidalgo              16709
México              101725
dtype: int64

Top activities:
 nombre_act
Comercio al por menor en tiendas de abarrotes, ultramarinos y misceláneas           157430
Comercio al por menor de frutas y verduras frescas                                   32763
Comercio al por menor de carne de aves                                               25637
Comercio al por menor de dulces y materias primas para repostería                    23163
Comercio al por menor de carnes rojas                                                19610
Comercio al por menor de otros alimentos                                             15660
Comercio al por menor de leche, otros pr

Download date for the database: 23 November 2026
DENUE Edition 05/2026
Files used: "denue_inegi_46111", "denue_inegi_46112"


Counts:
                Food retail (all)     | Tiendas de abarrotes
Mexico City            96,783         | 38,996
State of Mexico       201,680         | 101,725
Hidalgo                32,630         | 16,709
Total                 331,093         | 157,430

Abarrotes are half of all food and retail (7.5%)

In [3]:
import re
chains = ["OXXO", "7 ELEVEN", "7-ELEVEN", "SEVEN ELEVEN", "CIRCLE K", "KIOSKO", "EXTRA",
          "TIENDAS 3B", "TIENDA 3B", "BARA", "NETO", "BODEGA AURRERA", "WALMART EXPRESS",
          "SUPER CITY", "SUPERCITO", "SUPER KOMPRAS", "WALDOS", "MODELORAMA", "SIX"]
pattern = r"\b(" + "|".join(re.escape(c) for c in chains) + r")\b"
denue["chain"] = denue["nom_estab"].str.upper().str.extract(pattern, expand=False)

print("Chain stores found:", denue["chain"].notna().sum())
print(pd.crosstab(denue["chain"], denue["codigo_act"]).loc[:, lambda t: t.sum() > 0])

Chain stores found: 7568
codigo_act       461110  461121  461122  461130  461140  461150  461160  \
chain                                                                     
7 ELEVEN              0       0       0       0       0       0       0   
BARA                 15       0       0       3       1       1       0   
BODEGA AURRERA        3       0       0       0       0       0       0   
CIRCLE K              0       0       0       0       0       0       0   
EXTRA                 2       0       0       0       0       1       0   
KIOSKO               17       1       0       0       0       1       4   
MODELORAMA          102       0       0       0       0       0       0   
NETO                 40       1       2       1       1       2       1   
OXXO                 11       0       0       0       0       0       1   
SEVEN ELEVEN          1       0       0       0       0       0       1   
SIX                 181       0       0       0       0       2       0   


In [5]:
up = denue["nom_estab"].str.upper()
for word in [r"\b3B\b", r"TRES B", r"SUPERCITO", r"MI BODEGA", r"\bSIX\b"]:
    hits = denue[up.str.contains(word, regex=True, na=False)]
    print(f"\n{word}: {len(hits)} stores")
    print(hits[["nom_estab", "codigo_act", "municipio"]].sample(min(8, len(hits)), random_state=1).to_string(index=False))


\b3B\b: 37 stores
            nom_estab codigo_act           municipio
                   3B     461110 Naucalpan de Juárez
TIENDA 3B CHIPILTEPEC     462112             Acolman
         ABARROTES 3B     461110             Acolman
            TIENDA 3B     462112 Tulancingo de Bravo
          POLLERIA 3B     461122               Lerma
            TIENDA 3B     462112         Tepotzotlán
            TIENDA 3B     462112             Acatlán
           TIENDAS 3B     462112   Ixtapan de la Sal

TRES B: 1417 stores
     nom_estab codigo_act            municipio
TIENDAS TRES B     462112       Miguel Hidalgo
TIENDAS TRES B     462112    Gustavo A. Madero
TIENDAS TRES B     462112    Gustavo A. Madero
TIENDAS TRES B     462112 Atizapán de Zaragoza
TIENDAS TRES B     462112          Chicoloapan
TIENDAS TRES B     462112   Cuautitlán Izcalli
TIENDAS TRES B     462112              Texcoco
TIENDAS TRES B     462112       Nicolás Romero

SUPERCITO: 276 stores
                                     

In [13]:
up = denue["nom_estab"].str.upper().fillna("")
code = denue["codigo_act"]

strong = r"\b(OXXO|7 ?-?ELEVEN|SEVEN ELEVEN|CIRCLE K|TIENDAS? (3B|TRES B)|BODEGA AURRERA|MI BODEGA|WALMART EXPRESS|CHEDRAUI|SORIANA)\b"
ambiguous = r"\b(SUPERCITO|NETO|EXTRA|KIOSKO|BARA|SUPER CITY|SUPER KOMPRAS|WALDOS)\b"
beer = r"\b(SIX|MODELORAMA)\b"

is_chain = (up.str.contains(strong, regex=True) | (up.str.contains(ambiguous, regex=True) & code.isin(["462111", "462112"]))) & (code != "462111")
is_beer = up.str.contains(beer, regex=True) | (code == "461212")

denue["store_type"] = "other_food_retail"
denue.loc[code == "462111", "store_type"] = "supermarket"
denue.loc[code == "462112", "store_type"] = "independent_minisuper"
denue.loc[is_beer, "store_type"] = "beer_store"
denue.loc[is_chain, "store_type"] = "chain_store"
denue.loc[(code == "461110") & ~is_chain & ~is_beer, "store_type"] = "independent_abarrotes"   # our target

print(denue.groupby(["store_type", "entidad"]).size().unstack(fill_value=0))
denue.to_parquet(ROOT / "data" / "interim" / "denue_zmvm_retail.parquet", index=False)

C:\Users\nbeckers\AppData\Local\Temp\ipykernel_25880\23572841.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  is_chain = (up.str.contains(strong, regex=True) | (up.str.contains(ambiguous, regex=True) & code.isin(["462111", "462112"]))) & (code != "462111")
C:\Users\nbeckers\AppData\Local\Temp\ipykernel_25880\23572841.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  is_beer = up.str.contains(beer, regex=True) | (code == "461212")


entidad                Ciudad de México  Hidalgo  México
store_type                                              
beer_store                         1129      991    2611
chain_store                        2738      638    3327
independent_abarrotes             38900    16609  101605
independent_minisuper               335       73     403
other_food_retail                 53368    14262   93257
supermarket                         313       57     477


In [4]:
census_files = sorted(RAW.rglob("conjunto_de_datos/conjunto_de_datos_ageb_urbana_*_cpv2020.csv"))
census = pd.concat([pd.read_csv(f, dtype=str) for f in census_files], ignore_index=True)
print(census.shape)
print(list(census.columns)[:20])
print(census["NOM_LOC"].value_counts().head(6))

(253243, 230)
['ENTIDAD', 'NOM_ENT', 'MUN', 'NOM_MUN', 'LOC', 'NOM_LOC', 'AGEB', 'MZA', 'POBTOT', 'POBFEM', 'POBMAS', 'P_0A2', 'P_0A2_F', 'P_0A2_M', 'P_3YMAS', 'P_3YMAS_F', 'P_3YMAS_M', 'P_5YMAS', 'P_5YMAS_F', 'P_5YMAS_M']
NOM_LOC
Iztapalapa             15112
Ecatepec de Morelos    13571
Gustavo A. Madero       8690
Total AGEB urbana       8339
Naucalpan de Juárez     7682
Cuautitlán Izcalli      6432
Name: count, dtype: int64


In [8]:
ageb = census[census["NOM_LOC"] == "Total AGEB urbana"].copy()
ageb["cvegeo"] = ageb["ENTIDAD"].str.zfill(2) + ageb["MUN"].str.zfill(3) + ageb["LOC"].str.zfill(4) + ageb["AGEB"].str.zfill(4)
print(ageb.shape, "| unique AGEB keys:", ageb["cvegeo"].nunique())
print(ageb.groupby("NOM_ENT").size())
ageb.to_parquet(ROOT / "data" / "interim" / "census_ageb_2020.parquet", index=False)

(8339, 231) | unique AGEB keys: 8339
NOM_ENT
Ciudad de México    2433
Hidalgo             1441
México              4465
dtype: int64


In [14]:
%pip install xlrd

  Using cached xlrd-2.0.2-py2.py3-none-any.whl.metadata (3.5 kB)
Using cached xlrd-2.0.2-py2.py3-none-any.whl (96 kB)
Note: you may need to restart the kernel to use updated packages.


In [15]:
imu_path = RAW / "conapo" / "IMU_2020.xls"
xls = pd.ExcelFile(imu_path)
print("Sheets:", xls.sheet_names)
pd.read_excel(xls, sheet_name=0, header=None, nrows=8)

C:\Users\nbeckers\AppData\Local\Temp\ipykernel_25880\2058516105.py:2: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  xls = pd.ExcelFile(imu_path)


Sheets: ['Base de marginación 2020', 'IMU_2020', 'Diccionario']


,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,20,21,22
0,"Población total, indicadores socioeconómicos, ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Clave geográfica,Clave de la entidad federativa,Nombre de la entidad,Clave del municipio,Nombre del municipio,Clave de la localidad,Nombre de la localidad,Clave de la AGEB,Población total,% Población de 6 a 14 años que no asiste a la ...,...,% Ocupantes en viviendas particulares habitada...,% Ocupantes en viviendas particulares habitada...,% Ocupantes en viviendas particulares habitada...,% Ocupantes en viviendas particulares habitada...,% Ocupantes en viviendas particulares habitada...,% Ocupantes en viviendas particulares habitada...,% Ocupantes en viviendas particulares habitada...,"Índice de marginación, 2020","Grado de marginación, 2020","Índice de marginación normalizado, 2020"
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,CVE_AGEB,ENT,NOM_ENT,MUN,NOM_MUN,LOC,NOM_LOC,AGEB,POB_TOT,P6A14NAE,...,OVSEE,OVSAE,OVPT,OVHAC,OVSREF,OVSINT,OSCEL,IM_2020,GM_2020,IMN_2020
6,0100100010017,01,Aguascalientes,001,Aguascalientes,0001,Aguascalientes,0017,2237,13.076923,...,0.357942,0,0.089485,32.035794,19.239374,71.543624,3.44519,119.325886,Medio,0.932129
7,010010001006A,01,Aguascalientes,001,Aguascalientes,0001,Aguascalientes,006A,1411,0.8,...,0,0,0,2.05528,0,2.763997,0.566974,125.820757,Muy bajo,0.982865


In [16]:
imu = pd.read_excel(xls, sheet_name="Base de marginación 2020", header=5,
                    dtype={"CVE_AGEB": str, "ENT": str, "MUN": str, "LOC": str, "AGEB": str})
imu = imu[imu["ENT"].isin(states)].copy()
print(imu.shape)
print(imu["GM_2020"].value_counts())
imu.to_parquet(ROOT / "data" / "interim" / "conapo_imu_ageb_2020.parquet", index=False)

(7813, 23)
GM_2020
Bajo        2532
Medio       2470
Muy bajo    1279
Alto        1141
Muy alto     391
Name: count, dtype: int64


In [17]:
stores = denue[denue["store_type"] == "independent_abarrotes"].copy()
stores["cvegeo"] = (stores["cve_ent"].str.zfill(2) + stores["cve_mun"].str.zfill(3)
                    + stores["cve_loc"].str.zfill(4) + stores["ageb"].str.upper().str.zfill(4))

in_census = stores["cvegeo"].isin(ageb["cvegeo"])
in_imu = stores["cvegeo"].isin(imu["CVE_AGEB"])
print(f"Independent abarrotes: {len(stores):,}")
print(f"Matched to a Census AGEB: {in_census.mean():.1%}")
print(f"Matched to the CONAPO index: {in_imu.mean():.1%}")
print("\nUnmatched by state:\n", stores[~in_census].groupby("entidad").size())

Independent abarrotes: 157,114
Matched to a Census AGEB: 99.2%
Matched to the CONAPO index: 98.8%

Unmatched by state:
 entidad
Ciudad de México    437
Hidalgo              70
México              687
dtype: int64
